# Cours 4 "Traitement des données en SHS" - Analyses textuelles
### Louis Maritaud
### louis.maritaud@unilim.fr

## Objectifs pédagogiques
- Comprendre les licences de données et les entrepôts de données ouvertes
- Maîtriser les opérations sur les chaînes de caractères (strings)
- Utiliser les expressions régulières (regex) pour l'extraction de données
- Réaliser des traitements textométriques simples (nettoyage, stopwords, fréquences)

## Plan de séance

**1. Les entrepôts de données ouvertes & les licences**
- 1.1 Les licences de données
- 1.2 Recherche Data Gouv
- 1.3 Nakala
- 1.4 Gallica

**2. Utiliser un jeu de données dans Python**
- 2.1 Les fonctions adaptées aux textes
  - 2.1.1 Les expressions régulières (regex)
  - 2.1.2 Découpages automatiques avec split
- 2.2 Exemple de traitements textométriques simples
  - 2.2.1 TF-IDF
  - 2.2.2 Spacy, NLTK, classy-classification

# 1. Les entrepôts de données, les données ouvertes et les licences

## Données ouvertes ?
La science se construit de façon **collaborative**. Toute recherche ne nécessite pas nécessairement de récolter de nouvelles données, on peut tout à fait traiter des données **pré-existantes**.

On parle de _Science ouverte_ lorsque l'on respecte les prérogatives mises en place par le Ministère de l'Enseignement Supérieur et de la Recherche, qui consistent en ce que les données de la recherche, puisqu'elles sont récoltées sur des fonds publics, se doivent d'être **aussi ouvertes que possible, aussi fermées que nécessaire**.

En Histoire, il est assez rare de ne pas avoir le droit de produire des données ouvertes.

En règle générale, il s'agit d'une interdiction qui provient :
- → Soit du Règlement Général sur la Protection des Données personnelles (RGPD)
- → Soit de contraintes privées, comme pour les thèses CIFRE par exemple, ou les collaborations public/privé

Parfois dû à des contraintes de financeurs, on se retrouve avec de plus en plus de jeux de données ouverts et disponibles sur des espaces dédiés, les **entrepôts de données**.

## 1.1 Les licences de données

![IMG](DATA/Les-licences-CC.png)

Les licences Creative Commons permettent de définir les droits d'utilisation des données :
- **CC0** : Domaine public, aucune restriction
- **CC BY** : Attribution requise
- **CC BY-SA** : Attribution + partage dans les mêmes conditions
- **CC BY-NC** : Attribution + usage non commercial
- **CC BY-ND** : Attribution + pas de modification

## 1.2 Recherche Data Gouv
[Accéder à Recherche Data Gouv](https://entrepot.recherche.data.gouv.fr/)

Plateforme nationale pour le partage et la découverte de données de recherche.

## 1.3 Nakala
[Accéder à Nakala](https://nakala.fr/)

Service d'archivage pérenne pour les données de la recherche en SHS.

[Exemple d'un jeu de données sur les villas Gallo Romaines dont on va se servir plus tard](https://www.nakala.fr/10.34847/nkl.7f52y7e6)

## 1.4 Gallica
[Accéder à Gallica](https://gallica.bnf.fr/accueil/fr/html/accueil-fr)

Bibliothèque numérique de la BnF avec millions de documents numérisés.

# 2. Utiliser un jeu de données dans Python

## 2.1 Les fonctions adaptées aux textes

**Rappel** : Un texte en Python est un `string`, une chaîne de caractères. Elle est notée entre guillemets.

Sur les str, on peut opérer plusieurs manipulations simples :

| Méthode | Ce qu'elle fait | Exemple |
|---|---|---|
| `.lower()` | Met tout en minuscules | `"Bonjour".lower()` → `"bonjour"` |
| `.upper()` | Met tout en majuscules | `"Bonjour".upper()` → `"BONJOUR"` |
| `.strip()` | Supprime les espaces avant/après | `" Bonjour ".strip()` → `"Bonjour"` |
| `.split(séparateur)` | Découpe une chaîne en liste | `"a,b,c".split(",")` → `["a","b","c"]` |
| `.replace(ancien, nouveau)` | Remplace des morceaux de texte | `"chat".replace("ch","r")` → `"rat"` |
| `" ".join(liste)` | Colle une liste en chaîne | `" ".join(["a","b"])` → `"a b"` |
| `.count(mot)` | Compte les occurrences d'un mot | `"abab".count("ab")` → `2` |
| `.startswith(préfixe)` | Vérifie le début de la chaîne | `"Bonjour".startswith("Bon")` → `True` |
| `.endswith(suffixe)` | Vérifie la fin de la chaîne | `"Bonjour".endswith("jour")` → `True` |


Pour la suite, on va utiliser un corpus (défini de façon très aléatoire par votre serviteur) de 3 textes de Guillaume de Machaut, et qui datent du 14ème siècle. Ils sont compris dans la Base de français médiéval, corpus BFM 2019 et corpus BFM 2022.

[Le dit dou Lyon](https://nakala.fr/10.34847/nkl.f4bf60v0)

[Remede de Fortune](https://nakala.fr/10.34847/nkl.9cafuiw9)

[Le Lay de plour](https://nakala.fr/10.34847/nkl.2c8fz0ry)

Petit disclaimer, ce sont des textes déjà traités en TEI-XML. Je les ai transformés en .txt avec uniquement le texte sans annotations ni rien pour qu'on travaille dessus. Le code est en dessous, si vous voulez je vous explique (mais ce sera pas utile pour vous dans le cadre de ce cours je pense)

In [ ]:
from bs4 import BeautifulSoup
from glob import glob
import pandas as pd

def extraire_texte(xml):
    with open(xml, "r", encoding="utf-8") as f:
        texte_xml=f.read()
    soup= BeautifulSoup(texte_xml, 'xml')
    body=soup.find("body")

    texte=body.get_text(separator=" ", strip=True)
    return texte

for file in glob("DATA/XML/*.xml") :
    texte=extraire_texte(file)
    if "Fortune" in file:
        titre="Remede_de_Fortune.txt"
    elif "Lyon" in file:
        titre="Le_dit_dou_Lyon.txt"
    else:
        titre="Le_Lay_de_plour.txt"
    with open("DATA/texte/"+titre, 'w', encoding="utf-8") as out:
        out.write(texte)

## Maintenant, nos textes sont accessibles dans `DATA/texte/`

On va commencer par les mettre dans un dictionnaire ! Et vous allez le faire.

In [ ]:
# Avec une boucle for, vous allez itérer sur les différents fichiers dans le dossier DATA/texte/ grâce à la fonction "glob"
from glob import glob


# glob renvoie une liste des fichiers dans un dossier
fichiers = glob("DATA/texte/*.txt")

# fichiers est donc une liste qui comprend ici tous les fichiers dans le dossier texte qui ont une extension .txt
# On va les ouvrir et les lire, puis stocker leur contenu dans un dictionnaire.
# 1ere étape : faire un dictionnaire dont les valeurs sont des listes vides
dictionnaire = {
    "texte":[],
    # Complétez
}
# deuxième étape, avec une boucle for aller chercher les infos *

for texte in fichiers: # fichiers = la liste créé par glob
    with open(texte,"r", encoding="utf-8") as texte:
        contenu=texte.read() #le contenu de chaque fichier est lu et stocké dans la variable contenu
    # Complétez en dehors de l'identation de with
    print(contenu)

# Infos sur les textes : 
# Auteur : Guillaume de Machaut pour les trois
# Dates : Le dit dou lyon : 1342-1342 ; le Lay de plour : 1334-1366 ; Remede de Fortune : 1341-1341
# En premier lieu, on va concevoir et construire un dictionnaire : 


# Ensuite, on va itérer sur la liste, et ouvrir chaque fichier, lire le contenu, puis le stocker dans notre dictionnaire





In [ ]:
# Si on galère trop :
corpus = pd.read_csv("DATA/corpus.csv")
display(corpus)

In [ ]:
# Exploration du corpus : 
# Dimensions du corpus
print(f"Le corpus contient {len(corpus)} textes.")
print(f"Les colonnes sont : {list(corpus.columns)}")

# Auteurs présents
print(f"\nAuteurs dans le corpus : {corpus['auteur'].unique()}")

# Nombre de textes par auteur
print("\nNombre de textes par auteur :")
print(corpus['auteur'].value_counts())

## 2.1.1 Les expressions régulières (regex)

Les expressions régulières (ou **regex**) sont des motifs de recherche puissants qui permettent de :
- Rechercher des patterns complexes dans un texte
- Extraire des informations structurées (dates, emails, numéros...)
- Valider des formats de données
- Remplacer du texte de manière sophistiquée

### Syntaxe de base

| Pattern | Signification | Exemple |
|---------|---------------|----------|
| `.` | N'importe quel caractère | `a.c` → "abc", "a2c" |
| `*` | 0 ou plusieurs fois | `ab*c` → "ac", "abc", "abbbbc" |
| `+` | 1 ou plusieurs fois | `ab+c` → "abc", "abbbbc" |
| `?` | 0 ou 1 fois (optionnel) | `ab?c` → "ac", "abc" |
| `^` | Début de chaîne | `^Bonjour` → "Bonjour le monde" |
| `$` | Fin de chaîne | `monde$` → "Bonjour le monde" |
| `[]` | Un caractère parmi | `[aeiou]` → n'importe quelle voyelle |
| `[^]` | Aucun caractère parmi | `[^0-9]` → pas un chiffre |
| `\d` | Un chiffre | `\d+` → "123" |
| `\w` | Lettre ou chiffre | `\w+` → "mot123" |
| `\s` | Espace | `\s+` → espaces multiples |
| `{n}` | Exactement n fois | `\d{4}` → "2024" |
| `{n,m}` | Entre n et m fois | `\d{2,4}` → "12" ou "1234" |
| `()` | Groupe de capture | `(\d{2})/(\d{2})` → jour/mois |

Pour les \[Lettre en minuscule], quand on met la lettre en majuscule ça fait l'inverse :    
- `\s`: espace → `\S` : tout ce qui n'est pas un espace
- `\w` : caractère alphanumérique → `\W` : tout ce qui n'est pas alphanumérique
- `\d` : chiffre → `\D` : tout ce qui n'est pas un chiffre 

### Module `re` en Python

Python utilise le module `re` pour travailler avec les regex :

```python
import re

# Fonctions principales
re.search(pattern, texte)    # Cherche la première occurrence
re.findall(pattern, texte)   # Trouve toutes les occurrences
re.sub(pattern, repl, texte) # Remplace toutes les occurrences
re.match(pattern, texte)     # Vérifie si le début correspond
re.split(pattern, texte)     # Découpe selon un pattern
```

**Les groupes de capture**    
Quand on fait des regex, on peut définir des groupes de capture avec des parenthèses. Chaque groupe aura un index et sera renvoyé sous forme de tuple

In [ ]:
import re

test = "Bonjour je m'appelle Louis Maritaud j'ai 31 ans, je suis ingénieur de recherche pour l'université de Limoges"
pattern = r"([A-Z]\w+ +[A-Z]\w+).*?(\d+).*?([U-u]niversité.+[A-Z]\w+)" 
trouves=re.search(pattern, test)
print(trouves.group(0))
print(trouves.groups())
print(trouves.group(1))
print(trouves.group(2))
print(trouves.group(3))



In [ ]:
import re

# Exemple 1 : Extraire des dates
texte = "Les événements ont eu lieu le 15/03/1789 et le 14/07/1789."
dates = re.findall(r'\d{2}/\d{2}/\d{4}', texte)
print(f"Dates trouvées : {dates}")

# Exemple 2 : Extraire des années
texte = "Entre 1914 et 1918, puis de 1939 à 1945."
annees = re.findall(r'\d{4}', texte)
print(f"Années trouvées : {annees}")

# Exemple 3 : Nettoyer des numéros de téléphone
telephone = "Mon numéro est 06.12.34.56.78"  # Ceci est un faux numéro
numero_propre = re.sub(r'\D+', '', telephone)
print(f"Numéro nettoyé : {numero_propre}")

# Exemple 4 : Valider un email
email = "louis.maritaud@unilim.fr"
pattern_email = r'^[\w\.-]+@[\w\.-]+\.\w+$' 
if re.match(pattern_email, email):
    print(f"'{email}' est un email valide")

### Exercice 1 : Regex sur données historiques

Vous disposez d'un texte d'archives avec des dates et des montants. Utilisez les regex pour extraire ces informations.

In [ ]:
# Texte d'exemple
archive = """
Le 12 janvier 1715, le roi a versé 1500 livres tournois.
Le 23 mars 1715, nouvelle somme de 2300 livres.
Transaction du 5 avril 1716 : 850 livres tournois.
"""

# TODO : Extraire toutes les dates (format: jour mois année)
# Indice : \d+ pour les chiffres, \w+ pour les mots

# TODO : Extraire tous les montants en livres
# Indice : chercher "nombre + livres"

# Votre code ici

## Nettoyage d'un texte

On peut, à l'aide des **expressions régulières**, créer des **patterns** à rechercher dans un texte. Pattern signifie motif, en gros on va construire une forme de modèle semi abstrait qui sera ensuite testé sur tout le texte :

In [ ]:
import re  # bibliothèque pour les expressions régulières
import string # Pour retirer la ponctuation

def nettoyer_texte(texte):
    """
    Nettoie un texte : minuscules, suppression des accents problématiques,
    suppression de la ponctuation, suppression des espaces multiples.
    """
    # 1. Tout en minuscules
    texte = texte.lower()
    
    # 2. On garde uniquement les lettres, les espaces et les accents français
    #    re.sub(pattern, remplacement, texte) remplace tout ce qui correspond au pattern
    #    [^a-zéèêëàâùûîïôç1-9 ] signifie : tout ce qui N'EST PAS une lettre, un nombre, un accent ou un espace
    texte = re.sub(r'[^a-zéèêëàâùûîïôç1-9 ]', '', texte)
    
    # 3. On va supprimer toute la ponctuation aussi
    texte=texte.strip(string.punctuation)

    # 4. On supprime les espaces multiples (plusieurs espaces → un seul)
    texte = re.sub(r' +', ' ', texte)
    
    # 5. On supprime les espaces avant et après
    texte = texte.strip()

    return texte

# Test sur une phrase avec des "salissures" :
texte_sale = "  La LIBERTÉ, c'est LE droit   de TOUT citoyen !! (1789) "
texte_propre = nettoyer_texte(texte_sale)

print(f"Avant : '{texte_sale}'")
print(f"Après : '{texte_propre}'")

In [ ]:
#On applique notre fonction à nos textes dans le df :
corpus["texte_propre"]=corpus["texte"].apply(nettoyer_texte)
display(corpus)

## 2.1.2 Découpages automatiques avec split

## Tokenisation
La **tokenisation** (découpage en tokens = mots) est l'étape fondamentale de toute analyse textuelle.
On fait ça simplement avec `.split()` qui découpe sur les espaces.

# On va créer une nouvelle colonne qui contiendra des listes de mots

In [ ]:
corpus["tokens"]=corpus["texte_propre"].str.split()
display(corpus)

## Stop-words
Quand on compte les mots d'un texte, les plus fréquents sont souvent des mots très banaux : "le", "la", "de", "est"… On les appelle **stop words** (mots-outils). En analyse textuelle, on les retire pour se concentrer sur les mots **porteurs de sens**.

In [ ]:
# Liste de stop words français (simplifiée)
STOP_WORDS = {
    'le', 'la', 'les', 'un', 'une', 'des', 'du', 'de', 'et', 'en',
    'est', 'sont', 'que', 'qui', 'dans', 'par', 'pour', 'sur',
    'avec', 'ce', 'cette', 'ces', 'il', 'elle', 'nous', 'vous',
    'on', 'tout', 'tous', 'toute', 'toutes', 'même', 'mais',
    'donc', 'car', 'ni', 'ou', 'à', 'au', 'aux', 'pas', 'ne',
    'plus', 'moins', 'très', 'bien', 'aussi', 'comme',
    'sera', 'ont', 'ses', 'leur', 'leurs', 'mon', 'ma', 'mes', "a", "si", 
    "je", "le", "se", "sans",'sa', 'tu', 'cest', "me", "li", "te","son",
    "quant", 'tant', 'quil', 'moy', 'dont','quel', "quelle", 'fait', "com",
    "moult","grand", "einsi","faire","fu","trop","riens", "comment", "quen", "puis","y"}
# C'est un "set" (ensemble) : la recherche dans un set est très rapide


def filtrer_stop_words(liste_de_mots):
    """Retire les stop words d'une liste de mots."""
    return [mot for mot in liste_de_mots if mot not in STOP_WORDS and len(mot) > 3]


# Application sur notre df
corpus["token_propres"]=corpus["tokens"].apply(filtrer_stop_words)

# Et on ajoute une colonne qui contient le nombre de tokens finaux :
corpus["nb_tokens"]=corpus["token_propres"].str.len()
display(corpus)

## Exercice 2 : Appliquer le même traitement à Jean Renart

Vous allez maintenant traiter des textes de Jean Renart (début du 13ème siècle) de la même manière.

In [ ]:
# Création du corpus Jean Renart
dic = {
    "titre":[],
    "date_debut":[],
    "date_fin":[],
    "auteur":[],
    "texte":[]   
}
for file in glob("DATA/XML_e/*.xml"):

    texte=extraire_texte(file)
    if "ombre" in file:
        dic["titre"].append("Lai de l'ombre")
        dic["date_debut"].append(1217)
        dic["date_fin"].append(1222)
    elif "dole" in file:
        dic["titre"].append("Roman de la Rose ou de Guillaume de Dole")
        dic["date_debut"].append(1210)
        dic["date_fin"].append(1228)
    else:
        dic["titre"].append("Escoufle")
        dic["date_debut"].append(1200)
        dic["date_fin"].append(1202)
    dic["texte"].append(texte)
    dic["auteur"].append("Jean Renart")

corpus_jean= pd.DataFrame(dic)
display(corpus_jean)

In [ ]:
# Le df est désormais construit et stocké dans la variable corpus_jean, enrichissez les données et appliquez les traitements
corpus_jean["texte_propre"]=corpus_jean["texte"].apply(nettoyer_texte)
corpus_jean["tokens"]=corpus_jean["texte_propre"].str.split()
corpus_jean["token_propres"]=corpus_jean["tokens"].apply(filtrer_stop_words)
corpus_jean["nb_tokens"]=corpus_jean["token_propres"].str.len()
display(corpus_jean)
corpus_jean.to_csv("test.csv", encoding="utf-8", index=False)

## 2.2 Exemple de traitements textométriques simples

### Analyse de fréquence

Visualisons les mots les plus fréquents dans notre corpus de Jean Renart.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

# On met ensemble tous les mots du corpus :
all_words=[]
for texte in corpus_jean["token_propres"]:
    all_words.extend(texte)

comptage=Counter(all_words)
# On récupère les 12 mots les plus fréquents
top_mots = comptage.most_common(12)
mots_labels = [mot for mot, _ in top_mots]      # les noms (axe Y)
mots_nombres = [nombre for _, nombre in top_mots]  # les nombres (axe X)

# ── Création du graphique ──
fig, ax = plt.subplots(figsize=(10, 6))  # même syntaxe que GeoPandas

ax.barh(mots_labels, mots_nombres, color='steelblue')  # barh = barre horizontale

# On ajoute les valeurs à droite de chaque barre
for i, nombre in enumerate(mots_nombres):
    ax.text(nombre + 0.1, i, str(nombre), va='center', fontsize=10)

ax.set_xlabel('Nombre d\'occurrences', fontsize=12)
ax.set_title('12 mots les plus fréquents — Corpus de gros loveur du XIIIème porté sur la boisson', fontsize=14)
ax.invert_yaxis()  # Le mot le plus fréquent en haut

plt.tight_layout()
plt.show()

### Exercice 3 : Comparer les vocabulaires

Créez le même graphique pour le corpus de Guillaume de Machaut et comparez les résultats.

In [ ]:
# TODO :
# 1. Reprendre le corpus de Guillaume de Machaut
# 2. Compter les mots les plus fréquents
# 3. Créer un graphique
# 4. Comparer avec Jean Renart

# Votre code ici

## 2.2.1 TF-IDF (Term Frequency - Inverse Document Frequency)

Le TF-IDF permet de mesurer l'importance d'un mot dans un document par rapport à un corpus.

**Principe :**
- **TF** (fréquence du terme) : Combien de fois le mot apparaît dans le document
- **IDF** (inverse de la fréquence dans les documents) : Rareté du mot dans le corpus
- **TF-IDF** = TF × IDF : Les mots fréquents dans un document mais rares dans le corpus ont un score élevé

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Utiliser le texte propre (sans stopwords)
corpus_textes = [' '.join(tokens) for tokens in corpus_jean["token_propres"]]

# Créer la matrice TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus_textes)

# Récupérer les noms de features (mots)
feature_names = vectorizer.get_feature_names_out()

# Convertir en DataFrame pour visualisation
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names,
    index=corpus_jean["titre"]
)

print("Matrice TF-IDF (aperçu) :")
display(tfidf_df.iloc[:, :10])  # Afficher les 10 premiers mots

# Mots les plus importants par document
print("\nMots les plus significatifs par texte :")
for titre in corpus_jean["titre"]:
    top_words = tfidf_df.loc[titre].nlargest(5)
    print(f"\n{titre} :")
    for mot, score in top_words.items():
        print(f"  - {mot}: {score:.3f}")

## 2.2.2 Spacy, NLTK, classy-classification

### SpaCy
Bibliothèque de NLP (Natural Language Processing) moderne et performante.

**Fonctionnalités :**
- Tokenisation avancée
- Reconnaissance d'entités nommées (NER)
- Analyse syntaxique
- Lemmatisation

In [ ]:
# Installation (à faire une fois)
# !pip install spacy
!python -m spacy download fr_core_news_lg

In [ ]:
import spacy

# Charger le modèle français
nlp = spacy.load("fr_core_news_sm")

# Analyser un texte
texte = "Louis XIV régna sur la France de 1643 à 1715. Il vécut à Versailles."
doc = nlp(texte)

# Entités nommées
print("Entités nommées détectées :")
for ent in doc.ents:
    print(f"  - {ent.text} ({ent.label_})")

# Lemmatisation
print("\nLemmes :")
for token in doc:
    if not token.is_punct and not token.is_space:
        print(f"  {token.text} → {token.lemma_}")

### Exercice 4 : Extraction d'entités nommées sur le corpus médiéval

Utilisez SpaCy pour extraire automatiquement les noms de personnes et de lieux dans vos textes médiévaux.

In [ ]:
# TODO :
# 1. Charger le modèle SpaCy
# 2. Analyser un ou plusieurs textes du corpus
# 3. Extraire les entités de type PER (personne) et LOC (lieu)
# 4. Créer un DataFrame avec les résultats

# Votre code ici

### classy-classification : Classification de texte sans entraînement

**classy-classification** est une bibliothèque basée sur spaCy qui permet de classifier des textes **sans avoir besoin d'entraîner un modèle**. Elle utilise des embeddings (représentations vectorielles des mots) et le few-shot learning.

**Avantages :**
- Pas besoin de milliers d'exemples annotés
- Quelques exemples par catégorie suffisent
- Très utile pour les corpus historiques où les données annotées sont rares
- Rapide à mettre en place

**Cas d'usage en SHS :**
- Classifier des documents médiévaux par genre (chronique, poésie, traité)
- Identifier le ton d'un texte (laudatif, critique, neutre)
- Catégoriser des thèmes (amour courtois, guerre, religion)

In [ ]:
# Installation (à faire une fois)
!pip install classy-classification
# Le modèle SpaCy doit déjà être installé

#### Exemple 1 : Classification de genres littéraires

Nous allons classifier des extraits de textes médiévaux selon leur genre littéraire.

In [ ]:
import spacy
import classy_classification

# Charger le modèle français
nlp = spacy.load("fr_core_news_sm")

# Définir les catégories avec quelques exemples pour chacune
data = {
    "poésie_courtoise": [
        "Douce dame preuse et senée",
        "En qui j’ai mise ma pensée",
        "Et tout mon cuer entirement",
        "Je vous salu et me present",
        "A fere vostre volenté",
        "Comme cele qui me puet santé",
        "Doner et mort quand li plera" ,
        "Mes ja voz cuers tels ne sera",
        "Que de moi pité ne vous praingne",
        "L'amour me fait languir en douce mélancolie",
        "Rose vermeille, fleur de noblesse et d'honneur"
    ],
    "chronique_historique": [
        "A la loenge et à la gloire de la benoite et inseparable Trinité, Dieu Pere, Filz et Saint Esperit.",
        "Je qui à present sui comis de vraiement mettre en escript tous les faiz des roys de France regnans en mon temps, expose et met en françois la vie du glorieus roy monseigneur saint Loys",
        "Si comme le pere monseigneur saint Looys si volt aler en Aubigois, il laissa son reanme à garder à la royne Blanche sa fame, et ses enfanz, et s’en vint à la cité d’Avignon, et l’assist a grant force de gent.",
        "Tant les tint estroitement, et tant fist ruer perrieres et mangonniaus qu’il ne le porent endurer",
        "si se rendirent et se mistrent du tout à sa volenté"
    ],
    "roman_chevaleresque": [
        "Li boins roys Artus de Bretaigne",
        "La qui proeche nous ensengne",
        "Que nous soions preus et courtois",
        "Tint court si riche conme rois",
        "A chele feste qui tant couste",
        "C’on doit nonmer le Penthecouste",
        "Li rois fu a Cardœil en Gales",
        "Aprés mengier, parmi les sales",
        "Li chevalier s’atropelerent"
    ]
}

# Configurer le classificateur
nlp.add_pipe(
    "classy_classification",
    config={
        "data": data,
        "device": "cpu"
    }
)

# Tester sur de nouveaux textes
textes_test = [
    "Congié praing, à Dieu vous commant. Encore vous salu et remant Que vous me mandez et commandez Vo volenté, et entendez A moi geter de cest martire. Tout mon cuer ne vous puis escrire, Mes je pri Dieu si fetement Come je ne vous aim faintement, Que de vous m’envoit joie entiere. Douce dame oiez ma proiere.",
    "L’an en suivant après, par le conseil Pierre Mauclerc duc de Bretaigne et Hue le conte de la Marche descort mut entre le roy et les barons de France, et maintenoient les barons contre le roy, que la royne Blanche, sa mere, ne devoit pas gouverner si grant chose comme le royaume de France, et qu’il n’apartenoit pas à fame de tel chose faire. Et le roy maintenoit contre ses barons qu’il estoit assez puissanz de son reanme gouverner avoec l’aide des bonnes genz qui estoient de son conseil. Pour ceste chose murmurerent les barons et se mistrent en aguet comment il porroient avoir le roy par devers euls et tenir lei en leur garde et en leur seignorie.",
    "A l’uis de la chambre dehors Fu Dodinez et Sagremors, Li rois et mesire Gavains, Et si fu pres mesire Yvains, Et fu avec Calogrenans, Unz chevaliers mout avenans Qui lors out conmenchié .i. conte"
]

print("Classification des textes :\n")
for texte in textes_test:
    doc = nlp(texte)
    print(f"Texte : {texte}")
    print(f"Catégorie : {doc._.cats}")
    print(f"Catégorie la plus probable : {max(doc._.cats, key=doc._.cats.get)}")
    print("-" * 80)

#### Exemple 2 : Classification thématique du corpus de Guillaume de Machaut

Analysons les thèmes principaux des textes de notre corpus.

In [ ]:
pip install spacy-transformers

In [ ]:
# Créer un nouveau pipeline pour les thèmes
import spacy
from spacy.language import Language


nlp = spacy.load("fr_core_news_lg")

# Définir les thèmes avec des exemples
themes_data = {
    "amour": [
        "Douce dame preuse et senée",
        "En qui j’ai mise ma pensée",
        "Et tout mon cuer entirement",
        "Je vous salu et me present",
        "A fere vostre volenté",
        "Mes ja voz cuers tels ne sera",
        "Que de moi pité ne vous praingne",
        "Conment amours s’est prouvee",
        "Vers moi, qui tant l’ai amee",
    ],
    "nature": [
        "Ainz que la fueille descende",
        "Des arbres seur la ramee",
        "A l’entrant d’esté, que li tans conmence",
        "Que j’oi seur la flour les oisiauz tentir",
        "A l’entree de la saison"
    "Qu’ivers faut et lait le geler",
    "Que la flours naist lez le buisson"
    ],
    "mélancolie": [
       "Comme cele qui me puet santé",
        "Doner et mort quand li plera" ,
        "Que de moi pité ne vous praingne",
        "Et bel m’est, conment qu’il prende",
        "Que si bele mort aprende"

    ],
    "fortune": [
        "Vez cum Fortune le servi",
        "Qu’il ne se pot onques deffendre",
        "Qu’el nel’ féist au gibet pendre",
        "N’est-ce donc chose bien provable",
       "Que sa roé n’est pas tenable",
        "Que nus ne la puet retenir",
        "Tant sache à grant estat venir"
    ]
}

nlp_themes.add_pipe(
    "classy_classification",
    config={
        "data": themes_data,
        "model": "almanach/camembert-base",
        "device": "cpu"
    }
)


In [ ]:
# Analyser le corpus
resultats_themes = []

for idx, row in corpus.iterrows():
    for extrait_ in range(len(row['texte_propre'].split(" "))//10):
        extrait =" ".join(row['texte_propre'].split(" ")[extrait_*10:extrait_*10+10])
        doc = nlp_themes(extrait)
        theme_principal = max(doc._.cats, key=doc._.cats.get)
        score = doc._.cats[theme_principal]
        
        resultats_themes.append({
            'titre': row['titre'],
            'theme': theme_principal,
            'score': score,
            'tous_scores': doc._.cats
        })

df_themes = pd.DataFrame(resultats_themes)

print("\nThèmes identifiés dans le corpus de Guillaume de Machaut :\n")
display(df_themes[['titre', 'theme', 'score']])

display(df_themes[df_themes["score"]>0.5])

In [ ]:
# On va compter les valeurs pour chaque thème à l'intérieur de chaque oeuvre
grouped_df = df_themes.groupby("titre")["theme"].value_counts().reset_index()
#On réduit les colonnes
grouped_df.columns = ["titre", "theme", "count"]
display(grouped_df)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Pivoter le DataFrame : titres en lignes, thèmes en colonnes
pivot_df = grouped_df.pivot(index='titre', columns='theme', values='count').fillna(0)

# S'assurer que tous les thèmes sont présents (dans le bon ordre)
themes_list = list(themes_data.keys())
for theme in themes_list:
    if theme not in pivot_df.columns:
        pivot_df[theme] = 0
pivot_df = pivot_df[themes_list]  # réordonner les colonnes

# Préparer les données
titres = pivot_df.index.tolist()
scores_matrix = pivot_df.values.T  # shape: (nb_themes, nb_titres)

# Graphique
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(titres))
width = 0.6
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']
bottom = np.zeros(len(titres))

for i, theme in enumerate(themes_list):
    ax.bar(x, scores_matrix[i], width, label=theme, bottom=bottom, color=colors[i])
    bottom += scores_matrix[i]

ax.set_ylabel('Nombre de phrases classifiées', fontsize=12)
ax.set_title('Distribution thématique dans le corpus de Guillaume de Machaut', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(titres, rotation=45, ha='right')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

#### Visualisation des thèmes

### Exercice 5 : Créer votre propre classificateur

Créez un classificateur pour identifier un aspect dans des textes. Vous pouvre utiliser le classificateur sur les textes du corpus, dans l'exemple ci-dessous ça porte sur les émotions pour simplifier

In [ ]:
# TODO :
# 1. Définir 3-4 catégories émotionnelles
# 2. Fournir 3-4 exemples par catégorie
# 3. Configurer le classificateur
# 4. Tester sur votre corpus
# 5. Visualiser les résultats

# Votre code ici

# Exemple de structure :
# emotions_data = {
#     "joyeux": ["exemple 1", "exemple 2", "exemple 3"],
#     "triste": ["exemple 1", "exemple 2", "exemple 3"],
#     # etc.
# }

### Conseils pour classy-classification

**Pour obtenir de bons résultats :**

1. **Exemples de qualité** : Choisissez des exemples représentatifs et variés
2. **Nombre d'exemples** : 3-5 exemples par catégorie suffisent généralement
3. **Catégories distinctes** : Assurez-vous que les catégories sont bien différenciées
4. **Longueur des textes** : Fonctionne mieux sur des extraits courts à moyens (quelques phrases)
5. **Langue du modèle** : Utilisez un modèle multilingue pour le français

**Modèles recommandés :**
- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` (par défaut, bon compromis)
- `sentence-transformers/distiluse-base-multilingual-cased-v2` (plus rapide)
- `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` (plus précis mais plus lent)

## Conclusion

Dans ce cours, nous avons appris à :

1. **Trouver et utiliser des données ouvertes** en respectant les licences
2. **Manipuler du texte** avec les méthodes de strings et les regex
3. **Analyser des corpus médiévaux** avec des techniques textométriques

### Pour aller plus loin

- **Textométrie avancée** : Utilisez NLTK ou SpaCy pour des analyses plus poussées
- **Machine Learning** : Classez automatiquement vos textes avec scikit-learn

### Ressources

- [Documentation Pandas](https://pandas.pydata.org/docs/)
- [Regex101](https://regex101.com/) : Tester vos regex en ligne
- [SpaCy Documentation](https://spacy.io/)
- [Base de Français Médiéval](http://txm.bfm-corpus.org/)